# Scenario: AI-Powered Dev Workflow (Corporate Engineering Team)
# 🎯 Problem Statement
# In companies:
# - Developers manually write and test code.
# - Reviewers check for bugs and compliance.
# - Testing and validation are slow and error-prone.
# 👉 Solution
# A multi-agent system powered by Groq API where:
# - 👨‍💻 Coder Agent writes Python code for requested features.
# - 🔍 Reviewer Agent validates logic and ensures best practices.
# - 🧪 Tester Agent generates unit tests and executes them automatically.
# - 🔁 The loop continues until the code passes all tests and meets requirements.
# 🔄 Workflow
# Feature Request → Product Manager breaks into tasks → Coder writes code → Tester validates with unit tests → Reviewer checks compliance → Iteration loop → Final approved codebase
# 🧹 Conflict Resolution
# - Resolve conflicting dependencies (e.g., package version mismatches).
# - Standardize coding style across agents (PEP8 compliance).
# - Ensure Groq API integration avoids latency or model mismatch issues.


import autogen
import os
from google.colab import userdata

# Groq API configuration
# Make sure you have stored your Groq API key in Colab Secrets as 'GROQ_API_KEY'
groq_api_key = userdata.get("GROQ_API_KEY")

llm_config = {
    "config_list": [
        {
            "model": "llama-3.3-70b-versatile",   # Updated to a supported Groq model
            "api_key": groq_api_key,
            "base_url": "https://api.groq.com/openai/v1"  # Groq endpoint
        }
    ]
}

# Specialized agents
pm = autogen.AssistantAgent(
    name="ProductManager",
    llm_config=llm_config,
    system_message="You break down feature requests into tasks."
)

coder = autogen.AssistantAgent(
    name="Coder",
    llm_config=llm_config,
    system_message="You write Python code for given tasks."
)

tester = autogen.AssistantAgent(
    name="Tester",
    llm_config=llm_config,
    system_message="You write unit tests for code."
)

user = autogen.UserProxyAgent(
    name="User",
    human_input_mode="NEVER",
    code_execution_config={"work_dir": "project"}
)

# Group chat setup
groupchat = autogen.GroupChat(
    agents=[user, pm, coder, tester],
    messages=[],
    max_round=12,
    speaker_selection_method="auto"
)

# Manager orchestrates workflow
manager = autogen.GroupChatManager(
    groupchat=groupchat,
    llm_config=llm_config
)

# Start the group task
user.initiate_chat(manager,
    message="Build a simple REST API for a todo app")

In [3]:
# ============================================================
# Scenario: AI-Powered Dev Workflow (Corporate Engineering Team)
# Stable Version: JSON only for small metadata, tagged blocks for code/tests
# ============================================================

import os
import json
import re
from google.colab import userdata
from openai import OpenAI

# ------------------------------------------------------------
# 1. API Setup (Groq via OpenAI-compatible client)
# ------------------------------------------------------------
groq_api_key = userdata.get("GROQ_API_KEY")
os.environ["GROQ_API_KEY"] = groq_api_key

if not groq_api_key:
    raise ValueError(
        "GROQ_API_KEY not found in Colab secrets. "
        "Add it in the Secrets panel with key name: GROQ_API_KEY"
    )

client = OpenAI(
    api_key=groq_api_key,
    base_url="https://api.groq.com/openai/v1"
)

MODEL_NAME = "llama-3.3-70b-versatile"

# ------------------------------------------------------------
# 2. Feature Request
# ------------------------------------------------------------
feature_request = """
Build a Python module for employee task tracking.

Requirements:
1. Create a class TaskManager
2. Support:
   - add_task(title, owner, priority)
   - list_tasks()
   - mark_done(title)
   - get_pending_tasks()
3. Handle edge cases:
   - duplicate task titles
   - marking unknown task as done
   - empty task list
4. Use clean Python and follow PEP8
5. Include unit tests
6. Ensure package/dependency simplicity
"""

# ------------------------------------------------------------
# 3. Agent System Prompts
# ------------------------------------------------------------
PRODUCT_MANAGER_SYSTEM = """
You are the Product Manager Agent.

Your job:
- Read the feature request
- Break it into clear engineering tasks
- Define acceptance criteria
- Return ONLY valid JSON
- Do not use markdown
- Do not use triple backticks

Required JSON structure:
{
  "project_name": "",
  "tasks": ["task 1", "task 2"],
  "acceptance_criteria": ["criterion 1", "criterion 2"],
  "dependency_notes": ["note 1"]
}
"""

CODER_SYSTEM = """
You are the Coder Agent.

Your job:
- Write production-style Python code
- Keep it dependency-light
- Follow PEP8
- Handle edge cases
- Do not return JSON
- Do not use markdown fences
- Return exactly in this tag format:

<<MODULE_FILENAME>>
task_manager.py
<</MODULE_FILENAME>>

<<PYTHON_CODE>>
full python code here
<</PYTHON_CODE>>
"""

TESTER_SYSTEM = """
You are the Tester Agent.

Your job:
- Read the produced Python code
- Generate unit tests
- Simulate expected result
- Identify logic issues or missing edge cases
- Do not return JSON
- Do not use markdown fences
- Return exactly in this tag format:

<<TEST_FILENAME>>
test_task_manager.py
<</TEST_FILENAME>>

<<UNIT_TESTS>>
full unittest code here
<</UNIT_TESTS>>

<<SIMULATED_RESULT>>
PASS
<</SIMULATED_RESULT>>

<<ISSUES>>
issue 1
issue 2
<</ISSUES>>
"""

REVIEWER_SYSTEM = """
You are the Reviewer Agent.

Your job:
- Review implementation quality
- Check for PEP8 cleanliness
- Check edge-case handling
- Check dependency consistency
- Check whether code and tests meet requirements
- Do not return JSON
- Do not use markdown fences
- Return exactly in this tag format:

<<APPROVED>>
YES
<</APPROVED>>

<<FINAL_STATUS>>
APPROVED
<</FINAL_STATUS>>

<<REVIEW_COMMENTS>>
comment 1
comment 2
<</REVIEW_COMMENTS>>

<<REQUIRED_FIXES>>
fix 1
fix 2
<</REQUIRED_FIXES>>
"""

# ------------------------------------------------------------
# 4. Helper Functions
# ------------------------------------------------------------
def call_llm(system_prompt, user_prompt):
    response = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=0,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )
    return response.choices[0].message.content.strip()


def safe_json_load(text):
    text = text.strip()

    text = re.sub(r"^```json\s*", "", text)
    text = re.sub(r"^```\s*", "", text)
    text = re.sub(r"\s*```$", "", text)

    try:
        return json.loads(text)
    except Exception:
        start = text.find("{")
        end = text.rfind("}")
        if start != -1 and end != -1 and end > start:
            candidate = text[start:end + 1]
            candidate = re.sub(r"[\x00-\x1F\x7F]", "", candidate)
            return json.loads(candidate)

    raise ValueError(f"Response was not valid JSON:\n{text}")


def extract_tag(text, tag_name, required=True):
    pattern = rf"<<{tag_name}>>\s*(.*?)\s*<</{tag_name}>>"
    match = re.search(pattern, text, re.DOTALL)
    if match:
        return match.group(1).strip()
    if required:
        raise ValueError(f"Missing tag: {tag_name}\n\nRaw response:\n{text}")
    return ""


def parse_lines_block(block_text):
    lines = []
    for line in block_text.splitlines():
        cleaned = line.strip()
        if cleaned:
            lines.append(cleaned)
    return lines


def parse_coder_response(text):
    return {
        "module_filename": extract_tag(text, "MODULE_FILENAME"),
        "python_code": extract_tag(text, "PYTHON_CODE")
    }


def parse_tester_response(text):
    issues_block = extract_tag(text, "ISSUES", required=False)
    issues = parse_lines_block(issues_block) if issues_block else []

    return {
        "test_filename": extract_tag(text, "TEST_FILENAME"),
        "unit_tests": extract_tag(text, "UNIT_TESTS"),
        "simulated_test_result": extract_tag(text, "SIMULATED_RESULT").upper(),
        "issues": issues
    }


def parse_reviewer_response(text):
    comments_block = extract_tag(text, "REVIEW_COMMENTS", required=False)
    fixes_block = extract_tag(text, "REQUIRED_FIXES", required=False)

    approved_text = extract_tag(text, "APPROVED").strip().upper()
    approved = approved_text in {"YES", "TRUE", "APPROVED"}

    return {
        "approved": approved,
        "final_status": extract_tag(text, "FINAL_STATUS").strip().upper(),
        "review_comments": parse_lines_block(comments_block) if comments_block else [],
        "required_fixes": parse_lines_block(fixes_block) if fixes_block else []
    }


# ------------------------------------------------------------
# 5. Product Manager Stage
# ------------------------------------------------------------
pm_prompt = f"""
Feature Request:
{feature_request}

Break this into tasks and acceptance criteria.
Return valid JSON only.
"""

pm_raw = call_llm(PRODUCT_MANAGER_SYSTEM, pm_prompt)
pm_output = safe_json_load(pm_raw)

print("=" * 70)
print("PRODUCT MANAGER OUTPUT")
print("=" * 70)
print(json.dumps(pm_output, indent=2, ensure_ascii=False))

# ------------------------------------------------------------
# 6. Iterative Multi-Agent Workflow
# ------------------------------------------------------------
max_iterations = 4
coder_output = None
tester_output = None
reviewer_output = None
feedback = "No previous feedback."

for iteration in range(1, max_iterations + 1):
    print("\n" + "=" * 70)
    print(f"ITERATION {iteration}")
    print("=" * 70)

    # ---------------- Coder ----------------
    coder_prompt = f"""
Project breakdown:
{json.dumps(pm_output, indent=2)}

Feature request:
{feature_request}

Reviewer/Tester feedback from previous round:
{feedback}

Write the Python module.
Use the required tag format exactly.
"""
    coder_raw = call_llm(CODER_SYSTEM, coder_prompt)
    coder_output = parse_coder_response(coder_raw)

    print("\nCoder completed module draft.")
    print(f"Module file: {coder_output['module_filename']}")

    # ---------------- Tester ----------------
    tester_prompt = f"""
Feature request:
{feature_request}

Project breakdown:
{json.dumps(pm_output, indent=2)}

Python module filename:
{coder_output['module_filename']}

Python module code:
{coder_output['python_code']}

Generate unit tests and simulate whether the implementation should pass.
Use the required tag format exactly.
"""
    tester_raw = call_llm(TESTER_SYSTEM, tester_prompt)
    tester_output = parse_tester_response(tester_raw)

    print("Tester generated unit tests.")
    print(f"Test file: {tester_output['test_filename']}")
    print(f"Simulated result: {tester_output['simulated_test_result']}")

    # ---------------- Reviewer ----------------
    reviewer_prompt = f"""
Feature request:
{feature_request}

Project breakdown:
{json.dumps(pm_output, indent=2)}

Python module filename:
{coder_output['module_filename']}

Python module code:
{coder_output['python_code']}

Test filename:
{tester_output['test_filename']}

Unit tests:
{tester_output['unit_tests']}

Tester simulated result:
{tester_output['simulated_test_result']}

Tester issues:
{chr(10).join('- ' + x for x in tester_output['issues']) if tester_output['issues'] else 'None'}

Review code quality, test quality, dependency consistency, and engineering readiness.
Use the required tag format exactly.
"""
    reviewer_raw = call_llm(REVIEWER_SYSTEM, reviewer_prompt)
    reviewer_output = parse_reviewer_response(reviewer_raw)

    print("Reviewer completed review.")
    print(json.dumps(reviewer_output, indent=2, ensure_ascii=False))

    approved = reviewer_output.get("approved", False)
    tests_pass = tester_output.get("simulated_test_result", "").upper() == "PASS"

    if approved and tests_pass:
        print("\nFinal status: APPROVED")
        break

    fixes = reviewer_output.get("required_fixes", [])
    test_issues = tester_output.get("issues", [])
    combined_feedback = fixes + test_issues

    if combined_feedback:
        feedback = "Please revise the code using this feedback:\n" + "\n".join(
            [f"- {item}" for item in combined_feedback]
        )
    else:
        feedback = "Please improve edge cases, code clarity, and test completeness."

# ------------------------------------------------------------
# 7. Save Artifacts
# ------------------------------------------------------------
module_filename = coder_output.get("module_filename", "task_manager.py")
test_filename = tester_output.get("test_filename", "test_task_manager.py")

with open(module_filename, "w", encoding="utf-8") as f:
    f.write(coder_output["python_code"])

with open(test_filename, "w", encoding="utf-8") as f:
    f.write(tester_output["unit_tests"])

summary = {
    "feature_request": feature_request.strip(),
    "product_manager_output": pm_output,
    "coder_output": coder_output,
    "tester_output": tester_output,
    "reviewer_output": reviewer_output
}

with open("dev_workflow_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

with open("dev_workflow_summary.txt", "w", encoding="utf-8") as f:
    f.write("AI-Powered Dev Workflow Summary\n")
    f.write("=" * 60 + "\n\n")
    f.write("FEATURE REQUEST\n")
    f.write(feature_request.strip() + "\n\n")
    f.write("PRODUCT MANAGER OUTPUT\n")
    f.write(json.dumps(pm_output, indent=2, ensure_ascii=False) + "\n\n")
    f.write("CODER OUTPUT\n")
    f.write(coder_output["python_code"] + "\n\n")
    f.write("TESTER OUTPUT\n")
    f.write(tester_output["unit_tests"] + "\n\n")
    f.write("REVIEWER OUTPUT\n")
    f.write(json.dumps(reviewer_output, indent=2, ensure_ascii=False) + "\n")

print("\nSaved files:")
print(f"- {module_filename}")
print(f"- {test_filename}")
print("- dev_workflow_summary.json")
print("- dev_workflow_summary.txt")

print("\nDONE")

PRODUCT MANAGER OUTPUT
{
  "project_name": "Employee Task Tracking Module",
  "tasks": [
    "Create TaskManager class with required methods",
    "Implement add_task method with title, owner, and priority parameters",
    "Implement list_tasks method to display all tasks",
    "Implement mark_done method to mark a task as done by title",
    "Implement get_pending_tasks method to retrieve pending tasks",
    "Handle edge cases for duplicate task titles, marking unknown tasks, and empty task lists",
    "Write unit tests for all methods and edge cases",
    "Ensure code adheres to PEP8 and uses simple dependencies"
  ],
  "acceptance_criteria": [
    "TaskManager class can be instantiated without errors",
    "add_task method successfully adds new tasks with unique titles",
    "list_tasks method returns a list of all tasks",
    "mark_done method marks a task as done by title and raises an error for unknown tasks",
    "get_pending_tasks method returns a list of pending tasks",
    "E